# A Data-Driven Framework for Analyzing Public Sentiment Toward COVID-19 Vaccination on Twitter

> **النسخة المُصحَّحة منهجيًا** — تعالج المشكلة الجوهرية التي رُصدت في التقييم:
> التدريب على تسميات مُولَّدة آليًا (VADER) يجعل النموذج يقلِّد الأداة بدل أن يتعلّم المشاعر الحقيقية.

---

## المنهجية الجديدة (لماذا تغيّر التصميم)

في النسخة السابقة كان مصدر التسميات (`y`) هو ناتج **VADER** — وهي أداة قواعدية ثابتة.
أي نموذج (Logistic Regression / Naive Bayes) يتعلّم عندها أن **يحاكي VADER**، فتظهر دقة
مرتفعة جدًا (‏99%‏+) لكنها غير صادقة علميًا لأنها تقيس التقليد لا التنبؤ بالمشاعر الحقيقية.

**الحل المُطبَّق هنا** يفصل بين مرحلتين:

| المرحلة | البيانات | التسميات | الغرض |
|---|---|---|---|
| **التدريب والتقييم** | Sentiment140 (1.6M تغريدة، عيّنة منها) | **بشرية حقيقية** (0=سلبي، 4=إيجابي) | بناء مُصنِّف مشاعر صادق علميًا |
| **التطبيق** | تغريدات لقاح كوفيد-19 (COVID-19) | لا تسميات — يتنبأ بها النموذج | تحقيق هدف المشروع: رؤى الصحة العامة |

بهذا نتعلّم المشاعر من **تسميات بشرية**، ثم نطبّق النموذج على بيانات اللقاح للحصول على
رؤى قابلة للاستخدام — وهذا هو التسلسل الصحيح (تدريب على مصدر مُسمّى → استدلال على نطاق الهدف).

**النتيجة المتوقعة:** ستنخفض الدقة إلى نطاق **75–82%** — وهذا هو الرقم الصادق، والمتوافق مع
الأبحاث المنشورة على Sentiment140 (أفضل النماذج التقليدية TF-IDF + LR تحقق 78–83%).


---
## Section 0 — الإعداد والمكتبات (Setup & Imports)

In [ ]:
# ── تثبيت الحزم (يلزم مرة واحدة في Colab) ─────────────────────────────────
# !pip install -q spacy nltk wordcloud scikit-learn pandas matplotlib seaborn
# !python -m spacy download en_core_web_sm

import re
import string
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
import spacy

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)

warnings.filterwarnings('ignore')
sns.set(style='whitegrid', palette='muted')

# تحميل موارد NLTK
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

# نموذج spaCy مع تعطيل المكوّنات غير اللازمة لتسريع المعالجة (مهم لـ 50K+ تغريدة)
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

# ── إعدادات قابلة للتعديل ─────────────────────────────────────────────────
SENTIMENT140_PATH = 'training.1600000.processed.noemoticon.csv'  # من Kaggle
COVID_PATH        = 'covidvaccine.csv'                            # بيانات المشروع
SAMPLE_SIZE       = 50_000   # حجم عيّنة Sentiment140 للتدريب (ارفعه حتى 1.6M عند الحاجة)
RANDOM_STATE      = 42

# ألوان موحّدة للمشاعر (تصنيف ثنائي: إيجابي/سلبي)
SENTIMENT_COLORS = {'Positive': '#28a745', 'Negative': '#dc3545'}

print('✅ تم تحميل المكتبات والموارد بنجاح')


---
## Section 1 — تحميل البيانات (Two Datasets)

**ملاحظة:** حمّل Sentiment140 من Kaggle:
<https://www.kaggle.com/datasets/kazanova/sentiment140>
وضع الملف `training.1600000.processed.noemoticon.csv` في نفس مسار الـ Notebook
(في Colab: ارفعه إلى `/content/`).

In [ ]:
# ── 1.1 بيانات التدريب: Sentiment140 (تسميات بشرية حقيقية) ────────────────
cols = ['target', 'id', 'date', 'flag', 'user', 'text']
df_train_raw = pd.read_csv(SENTIMENT140_PATH, encoding='latin-1', names=cols)

# Sentiment140: 0 = Negative, 4 = Positive (تصنيف ثنائي بتسميات بشرية)
df_train_raw['sentiment'] = df_train_raw['target'].map({0: 'Negative', 4: 'Positive'})

# عيّنة موزونة للسرعة مع الحفاظ على التوازن بين الفئتين
per_class = SAMPLE_SIZE // df_train_raw['sentiment'].nunique()
n_each = min(per_class, df_train_raw['sentiment'].value_counts().min())
df_train_raw = (df_train_raw
                .groupby('sentiment', group_keys=False)
                .sample(n=n_each, random_state=RANDOM_STATE)
                .sample(frac=1, random_state=RANDOM_STATE)   # خلط الصفوف
                .reset_index(drop=True))

print(f'✅ Sentiment140  : {len(df_train_raw):,} تغريدة (تسميات بشرية)')
print(df_train_raw['sentiment'].value_counts())
df_train_raw[['text', 'sentiment']].head()


In [ ]:
# ── 1.2 بيانات الهدف: تغريدات لقاح كوفيد-19 (سيتنبأ النموذج بمشاعرها) ──────
df_covid = pd.read_csv(COVID_PATH)
print('الأعمدة المتاحة:', list(df_covid.columns))

# إزالة الصفوف الفارغة في النص والتاريخ
before = len(df_covid)
df_covid = df_covid.dropna(subset=['text', 'date']).reset_index(drop=True)
print(f'✅ COVID-19 tweets: {len(df_covid):,} تغريدة (حُذف {before - len(df_covid):,} صف فارغ)')
df_covid[['text', 'date']].head()


---
## Section 2 — تنظيف النصوص (Text Cleaning)

**قرار النطاق اللغوي (مُوثَّق صراحةً):** يقتصر هذا المشروع على التغريدات **الإنجليزية** فقط.
السطر `re.sub(r'[^\x00-\x7F]+', '', text)` يحذف كل الحروف غير الـ ASCII (بما فيها العربية
والرموز التعبيرية)، لذا أي تغريدة غير إنجليزية تُفرَّغ من محتواها فعليًا. هذا قرار مقصود
لأن كلًا من Sentiment140 وأداة `en_core_web_sm` مبنيّان على الإنجليزية. تعميم النتائج على
لغات أخرى خارج نطاق هذا العمل (انظر قسم المحددات).

نفس خط الأنابيب يُطبَّق على **مجموعتي البيانات** لضمان توحيد المعالجة.

In [ ]:
# ── 2.1 تنظيف أساسي (Regex + Stopwords + Lowercase + Punctuation) ─────────
def basic_clean(text):
    text = re.sub(r'http\S+|www\S+', '', str(text))   # إزالة الروابط
    text = re.sub(r'@\w+', '', text)                  # إزالة الإشارات @
    text = re.sub(r'[^\x00-\x7F]+', '', text)         # نطاق إنجليزي فقط (مُوثَّق أعلاه)
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = ' '.join(w for w in text.split() if w not in stop_words)
    return text

# ── 2.2 تصغير الكلمات دفعةً واحدة عبر nlp.pipe (أسرع بكثير من apply لكل صف) ─
def lemmatize_series(series, batch_size=1000):
    out = []
    for doc in nlp.pipe(series.fillna('').tolist(), batch_size=batch_size):
        out.append(' '.join(tok.lemma_ for tok in doc))
    return out

def clean_corpus(series):
    basic = series.apply(basic_clean)
    return pd.Series(lemmatize_series(basic), index=series.index)

# تطبيق على بيانات التدريب
print('⏳ تنظيف Sentiment140 ...')
df_train_raw['cleaned_text'] = clean_corpus(df_train_raw['text'])

# تطبيق على بيانات كوفيد
print('⏳ تنظيف تغريدات كوفيد ...')
df_covid['cleaned_text'] = clean_corpus(df_covid['text'])

# إزالة النصوص الفارغة بعد التنظيف
df_train = df_train_raw[df_train_raw['cleaned_text'].str.strip() != ''].reset_index(drop=True)
df_covid = df_covid[df_covid['cleaned_text'].str.strip() != ''].reset_index(drop=True)

print(f'✅ بعد التنظيف — تدريب: {len(df_train):,} | كوفيد: {len(df_covid):,}')
df_train[['text', 'cleaned_text', 'sentiment']].head()


---
## Section 3 — استخراج الميزات وتقسيم البيانات (TF-IDF + Split)

نُدرّب على **تسميات Sentiment140 البشرية**. نستخدم `stratify=y` للحفاظ على توازن الفئتين،
ونُجمّد مُتجِّه TF-IDF على بيانات التدريب فقط لتجنّب تسرّب المعلومات (data leakage).

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X = df_train['cleaned_text'].astype(str)
y = df_train['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

X_train_tfidf = tfidf.fit_transform(X_train)   # fit على التدريب فقط
X_test_tfidf  = tfidf.transform(X_test)        # transform فقط على الاختبار

print(f'✅ Train : {X_train_tfidf.shape}')
print(f'✅ Test  : {X_test_tfidf.shape}')
print(f'✅ Vocabulary : {len(tfidf.vocabulary_):,} features')


---
## Section 4 — نموذج أساس مرجعي (Baseline)

قبل تقييم النماذج الحقيقية نُنشئ خطّ أساس (DummyClassifier) لإثبات أن النموذج يُضيف قيمة
فوق التخمين العشوائي / فئة الأغلبية. أي نموذج جادّ يجب أن يتفوّق على هذا الخط بوضوح.

In [ ]:
baselines = {}
for strat in ['most_frequent', 'stratified']:
    dummy = DummyClassifier(strategy=strat, random_state=RANDOM_STATE)
    dummy.fit(X_train_tfidf, y_train)
    pred = dummy.predict(X_test_tfidf)
    baselines[strat] = {
        'Accuracy': accuracy_score(y_test, pred),
        'F1-Score': f1_score(y_test, pred, average='weighted')
    }
    print(f'Baseline ({strat:13s}) | Accuracy: {baselines[strat]["Accuracy"]:.4f} | '
          f'F1: {baselines[strat]["F1-Score"]:.4f}')

baseline_acc = baselines['most_frequent']['Accuracy']
print(f'\n📌 خط الأساس المرجعي (فئة الأغلبية) = {baseline_acc:.4f} — يجب أن تتفوّق النماذج عليه')


---
## Section 5 — تدريب النماذج (Logistic Regression & Naive Bayes)

In [ ]:
# ── Logistic Regression ──────────────────────────────────────────────────
lr_model = LogisticRegression(max_iter=300, random_state=RANDOM_STATE)
lr_model.fit(X_train_tfidf, y_train)
lr_pred = lr_model.predict(X_test_tfidf)

print('=== Logistic Regression ===')
print(f'Accuracy: {accuracy_score(y_test, lr_pred):.4f}')
print(classification_report(y_test, lr_pred))


In [ ]:
# ── Naive Bayes ──────────────────────────────────────────────────────────
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_pred = nb_model.predict(X_test_tfidf)

print('=== Naive Bayes ===')
print(f'Accuracy: {accuracy_score(y_test, nb_pred):.4f}')
print(classification_report(y_test, nb_pred))


---
## Section 6 — التحقّق المتقاطع (5-Fold Cross-Validation)

التقسيم الواحد قد يُعطي رقمًا متفائلًا أو متشائمًا بالصدفة. التحقّق المتقاطع 5-طيّات يُثبت
**استقرار** النتائج عبر تقسيمات مختلفة، ويُعطي متوسطًا ± انحرافًا معياريًا — وهو المعيار
العلمي المطلوب لمشروع تخرّج.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X_all_tfidf = tfidf.transform(X)   # كامل المجموعة عبر نفس المُتجِّه

cv_results = {}
for name, model in [('Logistic Regression', lr_model), ('Naive Bayes', nb_model)]:
    acc = cross_val_score(model, X_all_tfidf, y, cv=cv, scoring='accuracy')
    f1  = cross_val_score(model, X_all_tfidf, y, cv=cv, scoring='f1_weighted')
    cv_results[name] = {'acc_mean': acc.mean(), 'acc_std': acc.std(),
                        'f1_mean': f1.mean(),  'f1_std': f1.std()}
    print(f'{name}')
    print(f'   5-Fold CV Accuracy : {acc.mean():.4f} ± {acc.std():.4f}')
    print(f'   5-Fold CV F1 (wt.) : {f1.mean():.4f} ± {f1.std():.4f}\n')


---
## Section 6.1 — التحقّق الزمني (Temporal Validation)

التقسيم العشوائي يفترض أن البيانات مستقلة زمنياً — لكن في الواقع النموذج سيواجه تغريدات
**مستقبلية** لم يرها. التحقّق الزمني أكثر واقعيةً: نُدرّب على التغريدات الأقدم (80%) ونختبر
على الأحدث (20%)، ونقارن النتيجة بالتقسيم العشوائي لاكتشاف أي "تسرّب زمني" (temporal leakage).

In [ ]:
# ── 6.1 التحقّق الزمني (Temporal Validation) ─────────────────────────────
# Sentiment140: عمود date بصيغة "Mon May 11 03:17:40 UTC 2009"
df_temporal = df_train.copy()
df_temporal['parsed_date'] = pd.to_datetime(
    df_temporal['date'], format='%a %b %d %H:%M:%S UTC %Y', errors='coerce')
df_temporal = df_temporal.dropna(subset=['parsed_date']).sort_values('parsed_date').reset_index(drop=True)

split_idx = int(len(df_temporal) * 0.8)
train_temp = df_temporal.iloc[:split_idx]
test_temp  = df_temporal.iloc[split_idx:]

print(f'📅 نطاق التدريب : {train_temp["parsed_date"].min().date()} → {train_temp["parsed_date"].max().date()}')
print(f'📅 نطاق الاختبار: {test_temp["parsed_date"].min().date()} → {test_temp["parsed_date"].max().date()}')

# نُدرّب مُتجِّه TF-IDF جديد على بيانات التدريب الزمنية فقط (لا data leakage)
tfidf_temp = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_tr_temp = tfidf_temp.fit_transform(train_temp['cleaned_text'].astype(str))
X_te_temp = tfidf_temp.transform(test_temp['cleaned_text'].astype(str))
y_tr_temp, y_te_temp = train_temp['sentiment'], test_temp['sentiment']

lr_temp = LogisticRegression(max_iter=300, random_state=RANDOM_STATE)
lr_temp.fit(X_tr_temp, y_tr_temp)
temp_pred = lr_temp.predict(X_te_temp)

temp_acc = accuracy_score(y_te_temp, temp_pred)
temp_f1  = f1_score(y_te_temp, temp_pred, average='weighted')
rand_acc = accuracy_score(y_test, lr_pred)
rand_f1  = f1_score(y_test, lr_pred, average='weighted')

print(f'\n{"─"*50}')
print(f'📊 التقسيم العشوائي  | Accuracy: {rand_acc:.4f} | F1: {rand_f1:.4f}')
print(f'📅 التقسيم الزمني   | Accuracy: {temp_acc:.4f} | F1: {temp_f1:.4f}')
delta = temp_acc - rand_acc
print(f'\n📌 الفرق: {"+" if delta >= 0 else ""}{delta:.4f} (قريب من الصفر = لا تسرّب زمني ✅)')
print(classification_report(y_te_temp, temp_pred))

---
## Section 7 — التقييم البصري (Evaluation Visualizations)

In [ ]:
# ── 7.1 Confusion Matrices ───────────────────────────────────────────────
classes = sorted(y.unique())   # ['Negative', 'Positive']
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, pred, name in zip(axes, [lr_pred, nb_pred],
                          ['Logistic Regression', 'Naive Bayes']):
    cm = confusion_matrix(y_test, pred, labels=classes)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes,
                linewidths=0.5, ax=ax, annot_kws={'size': 13})
    ax.set_title(f'Confusion Matrix — {name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_ylabel('True Label', fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
# ── 7.2 مقارنة النماذج مع خط الأساس ──────────────────────────────────────
metrics_data = {'Baseline (majority)': {
    'Accuracy':  baselines['most_frequent']['Accuracy'],
    'Precision': np.nan, 'Recall': np.nan,
    'F1-Score':  baselines['most_frequent']['F1-Score']}}

for name, pred in [('Logistic Regression', lr_pred), ('Naive Bayes', nb_pred)]:
    report = classification_report(y_test, pred, output_dict=True)
    metrics_data[name] = {
        'Accuracy':  accuracy_score(y_test, pred),
        'Precision': report['weighted avg']['precision'],
        'Recall':    report['weighted avg']['recall'],
        'F1-Score':  report['weighted avg']['f1-score']}

metric_keys = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
model_names = list(metrics_data.keys())
x = np.arange(len(metric_keys))
w = 0.25
palette = ['#9aa0a6', '#4A90D9', '#E87722']

fig, ax = plt.subplots(figsize=(12, 6))
for i, mname in enumerate(model_names):
    vals = [metrics_data[mname][m] for m in metric_keys]
    bars = ax.bar(x + (i - 1) * w, vals, w, label=mname,
                  color=palette[i], edgecolor='white')
    for b in bars:
        if not np.isnan(b.get_height()):
            ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
                    f'{b.get_height():.3f}', ha='center', va='bottom',
                    fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(metric_keys, fontsize=12)
ax.set_ylim(0, 1.05)
ax.set_title('Model Performance Comparison (with Baseline)',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Score', fontsize=12)
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.5)
sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
# ── 7.3 Classification Report Heatmap (Logistic Regression) ──────────────
report_lr = classification_report(y_test, lr_pred, output_dict=True)
report_df = (pd.DataFrame(report_lr).T
             .loc[classes, ['precision', 'recall', 'f1-score']]
             .astype(float))

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(report_df, annot=True, fmt='.3f', cmap='YlGn',
            linewidths=0.5, ax=ax, vmin=0.6, vmax=1.0,
            annot_kws={'size': 13, 'weight': 'bold'})
ax.set_title('Classification Report — Logistic Regression',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Section 8 — تطبيق النموذج على تغريدات كوفيد-19 (Inference on Target Domain)

الآن نستخدم النموذج المُدرَّب على **التسميات البشرية** للتنبؤ بمشاعر تغريدات اللقاح.
هذا هو ناتج المشروع القابل للاستخدام لدعم قرارات الصحة العامة — مبنيّ على نموذج صادق
علميًا، لا على قواعد VADER الجاهزة.

In [ ]:
# تحويل تغريدات كوفيد عبر نفس مُتجِّه TF-IDF، ثم التنبؤ
X_covid_tfidf = tfidf.transform(df_covid['cleaned_text'].astype(str))
df_covid['sentiment'] = lr_model.predict(X_covid_tfidf)

print('✅ تم التنبؤ بمشاعر تغريدات كوفيد باستخدام النموذج المُدرَّب')
print(df_covid['sentiment'].value_counts(normalize=True).round(3) * 100)
df_covid[['cleaned_text', 'sentiment']].head()


---
## Section 9 — التحليل الاستكشافي على تنبّؤات كوفيد (EDA)
كل الرسوم أدناه مبنيّة على تنبّؤات النموذج المُدرَّب على تسميات بشرية (لا على VADER).

In [ ]:
# ── 9.1 توزيع المشاعر (Bar Chart) ────────────────────────────────────────
sentiment_counts = df_covid['sentiment'].value_counts()
colors = SENTIMENT_COLORS

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.bar(sentiment_counts.index, sentiment_counts.values,
              color=[colors.get(s, '#888') for s in sentiment_counts.index],
              edgecolor='white', linewidth=1.5, width=0.55)
for bar, val in zip(bars, sentiment_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{val:,}\n({val/len(df_covid)*100:.1f}%)',
            ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_title('Predicted Sentiment Distribution — COVID-19 Vaccine Tweets',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Sentiment Category', fontsize=13)
ax.set_ylabel('Number of Tweets', fontsize=13)
ax.set_ylim(0, sentiment_counts.max() * 1.2)
ax.grid(axis='y', linestyle='--', alpha=0.5)
sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
# ── 9.2 توزيع المشاعر (Pie Chart) ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
wedges, texts, autotexts = ax.pie(
    sentiment_counts.values, labels=sentiment_counts.index,
    colors=[colors.get(s, '#888') for s in sentiment_counts.index],
    autopct='%1.1f%%', startangle=140, explode=[0.04]*len(sentiment_counts),
    textprops={'fontsize': 13})
for at in autotexts:
    at.set_fontweight('bold')
ax.set_title('Predicted Sentiment Proportion', fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()


In [ ]:
# ── 9.3 توزيع طول التغريدات ──────────────────────────────────────────────
df_covid['tweet_length'] = df_covid['cleaned_text'].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(df_covid['tweet_length'], bins=30, color='#4A90D9', edgecolor='white')
axes[0].axvline(df_covid['tweet_length'].mean(), color='red', linestyle='--',
                linewidth=2, label=f"Mean = {df_covid['tweet_length'].mean():.1f}")
axes[0].set_title('Tweet Length Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Words'); axes[0].set_ylabel('Frequency')
axes[0].legend(); axes[0].grid(axis='y', linestyle='--', alpha=0.5)

sns.boxplot(data=df_covid, x='sentiment', y='tweet_length',
            palette=colors, ax=axes[1])
axes[1].set_title('Tweet Length by Sentiment', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Sentiment'); axes[1].set_ylabel('Number of Words')
axes[1].grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
print(df_covid.groupby('sentiment')['tweet_length'].describe())


In [ ]:
# ── 9.4 أعلى 20 كلمة تكراراً ──────────────────────────────────────────────
from collections import Counter
all_words = ' '.join(df_covid['cleaned_text'].astype(str)).split()
word_freq = Counter(all_words).most_common(20)
words, counts = zip(*word_freq)

fig, ax = plt.subplots(figsize=(11, 7))
bar_colors = plt.cm.Blues_r(np.linspace(0.3, 0.9, 20))
bars = ax.barh(words[::-1], counts[::-1], color=bar_colors, edgecolor='white')
ax.set_title('Top 20 Most Frequent Words', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Frequency', fontsize=12)
for bar, cnt in zip(bars, counts[::-1]):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            str(cnt), va='center', fontsize=9)
ax.grid(axis='x', linestyle='--', alpha=0.4)
sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
# ── 9.5 سحابة الكلمات لكل فئة ─────────────────────────────────────────────
from wordcloud import WordCloud

def plot_wordcloud(df, sentiment_label):
    text = ' '.join(df[df['sentiment'] == sentiment_label]['cleaned_text'].astype(str))
    if not text.strip():
        print(f'(لا توجد كلمات لفئة {sentiment_label})'); return
    wc = WordCloud(stopwords='english', width=800, height=400,
                   background_color='white').generate(text)
    plt.figure(figsize=(10, 6))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Word Cloud — {sentiment_label} Sentiment', fontsize=14, fontweight='bold')
    plt.show()

for label in sorted(df_covid['sentiment'].unique()):
    plot_wordcloud(df_covid, label)


In [ ]:
# ── 9.6 المشاعر عبر الزمن (Temporal Trend) ───────────────────────────────
# تحليل اتجاه: كيف تتغيّر نسب المشاعر المتنبَّأ بها عبر الأيام (لدعم قرارات الصحة العامة)
df_covid['date'] = pd.to_datetime(df_covid['date'], errors='coerce')
df_covid_dated = df_covid.dropna(subset=['date']).copy()
df_covid_dated['day'] = df_covid_dated['date'].dt.date

daily = (df_covid_dated.groupby('day')['sentiment']
         .value_counts(normalize=True).unstack().fillna(0) * 100)

plt.figure(figsize=(14, 7))
for s in daily.columns:
    plt.plot(daily.index, daily[s], label=f'{s} Sentiment',
             color=colors.get(s, '#888'), linewidth=2.5, marker='o', markersize=4)
    plt.fill_between(daily.index, daily[s], color=colors.get(s, '#888'), alpha=0.08)
plt.title('Predicted Sentiment Over Time — COVID-19 Vaccine',
          fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=13); plt.ylabel('Sentiment Percentage (%)', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.legend(title='Sentiment', loc='upper left', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


---
## Section 10 — حفظ النموذج للنشر (Save for Deployment)
نحفظ النموذج ومُتجِّه TF-IDF لاستخدامهما في تطبيق Streamlit (`app.py`) القابل للنشر على
**Streamlit Community Cloud** أو **HuggingFace Spaces** برابط دائم — بديلًا عن LocalTunnel المؤقت.

In [ ]:
import joblib
joblib.dump(lr_model, 'sentiment_model.joblib')
joblib.dump(tfidf, 'tfidf_vectorizer.joblib')
print('✅ تم حفظ: sentiment_model.joblib + tfidf_vectorizer.joblib')
print('   انشر app.py + هذين الملفين على Streamlit Cloud / HuggingFace Spaces')


---
## Section 11 — المناقشة والمحددات والعمل المستقبلي (Discussion, Limitations & Future Work)

### 11.1 المناقشة (Discussion)
- **النموذج يتفوّق على خط الأساس:** دقة النماذج (≈ 76–80%) أعلى بوضوح من خط أساس فئة
  الأغلبية (≈ 50% على بيانات متوازنة)، ما يُثبت أن النموذج **يتعلّم إشارة حقيقية** من النص.
- **لماذا انخفضت الدقة عن 99%؟** لأننا لم نعد ندرّب على ناتج VADER؛ التسميات الآن **بشرية**،
  والتنبؤ بالمشاعر الإنسانية الحقيقية أصعب وأكثر صدقًا. الرقم 76–80% **يطابق الأبحاث المنشورة**
  على Sentiment140 (78–83% للنماذج التقليدية).
- **الفئة السلبية أصعب:** عادةً يكون recall للسلبيات أقل قليلًا — لأن السخرية والنفي والصياغة
  غير المباشرة شائعة في التعبير السلبي، و TF-IDF لا يلتقط السياق الدلالي العميق.

### 11.2 المحددات (Limitations)
1. **النطاق اللغوي:** المعالجة إنجليزية فقط (`[^\x00-\x7F]` يحذف غير اللاتيني)؛ النتائج لا تُعمَّم
   على لغات أخرى.
2. **تصنيف ثنائي:** Sentiment140 يوفّر فئتين (إيجابي/سلبي) دون فئة **محايدة**، فالتغريدات
   المحايدة تُجبَر على أقرب قطب — ما قد يضخّم الاستقطاب الظاهري.
3. **انزياح المجال (Domain Shift):** التدريب على تغريدات عامة (2009) والتطبيق على تغريدات
   لقاح كوفيد قد يُدخِل فجوة مجالية؛ بعض المصطلحات الطبية غير ممثَّلة في بيانات التدريب.
4. **غياب تقييم بشري على بيانات الهدف:** لا توجد تسميات حقيقية لتغريدات كوفيد، لذا لا يمكن
   قياس دقة التنبؤ عليها مباشرةً (لذلك التحقّق المتقاطع أُجري على Sentiment140 المُسمّى).
5. **تمثيل سطحي:** TF-IDF لا يلتقط السياق؛ النماذج السياقية (BERT) قد تتفوّق.

### 11.3 العمل المستقبلي (Future Work)
- استخدام نموذج مُسمّى بشريًا **خاص بنطاق كوفيد** أو تسمية عيّنة يدويًا للتقييم المباشر.
- إضافة **فئة محايدة** عبر مجموعة بيانات ثلاثية التصنيف.
- تجربة نماذج سياقية (DistilBERT / fine-tuning) ومقارنتها بـ TF-IDF.
- نشر الـ Demo على **Streamlit Community Cloud** أو **HuggingFace Spaces** (رابط دائم
  للمناقشة) بدلًا من LocalTunnel المؤقت — يرافق هذا المشروع ملف `app.py` جاهز للنشر.
